In [16]:
from pathlib import Path

for p in Path("configs").glob("**/*"):
    print(p)

configs/config.yaml
configs/experiments.yaml
configs/final_full.yaml
configs/finalists.yaml
configs/legacy_500hz.yaml
configs/local.yaml
configs/medium.yaml
configs/research.yaml
configs/selected_study.yaml
configs/study.yaml
configs/training.yaml


In [18]:
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

print(cfg.keys())

dict_keys(['schema', 'study_id', 'run_mode', 'data', 'cohorts', 'preprocessing', 'features', 'model', 'training', 'evaluation', 'paths'])


In [20]:
from pathlib import Path
from data import prepare
from config import load_config

cfg = load_config("configs/config.yaml")

cache, manifest = prepare(cfg)

print("cache:", cache)
print("splits:", manifest["splits"])

ImportError: cannot import name 'cohort_limits' from 'config' (/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src/config.py)

In [21]:
import config

print(config.__file__)

/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src/config.py


In [22]:
import inspect
import config

print(inspect.getsourcefile(config))

/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src/config.py


In [23]:
import re

with open("src/config.py") as f:
    text = f.read()

print("cohort_limits" in text)

True


In [24]:
from config import cohort_limits

print(cohort_limits)


ImportError: cannot import name 'cohort_limits' from 'config' (/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src/config.py)

In [25]:
with open("src/config.py") as f:
    text = f.read()

idx = text.find("cohort_limits")

print(text[idx-300:idx+500])

"cuda" if torch.cuda.is_available() else "cpu") if requested == "auto" else requested
    return torch.device(device)


def run_directory(cfg, architecture, variant):
    return Path(cfg["paths"]["experiments"]) / cfg.get("experiment", {}).get(
        "name", f"{architecture}_{variant}"
    )
    
cohort_limits = {
    "pilot": {"train": 300, "val": 75},
    "medium": {"train": 1500, "val": None},
    "full": {"train": None, "val": None},
}

def file_hash(path):
    """Compute SHA256 hash of a file."""
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def source_hashes():
    """Return SHA256 hashes of source python files."""
    source_root = Path(__file__).parent
    retu


In [26]:
import importlib
import config

importlib.reload(config)

print(hasattr(config, "cohort_limits"))
print(config.cohort_limits)

True
{'pilot': {'train': 300, 'val': 75}, 'medium': {'train': 1500, 'val': None}, 'full': {'train': None, 'val': None}}


In [28]:
from data import prepare

In [29]:
cache, manifest = prepare(cfg)

print(type(cache))
print(manifest.keys())

FileNotFoundError: [Errno 2] No such file or directory: 'PTB-XL/ptbxl_database.csv'

In [30]:
from pathlib import Path

print(Path("PTB-XL").resolve())
print(Path("PTB-XL").exists())

/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/PTB-XL
False


In [31]:
from pathlib import Path

ptb = Path("/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL")

print(ptb.exists())
print((ptb / "ptbxl_database.csv").exists())
print((ptb / "scp_statements.csv").exists())
print((ptb / "records100").exists())

True
True
True
True


In [32]:
cfg["data"]["root"] = str(ptb)

cache, manifest = prepare(cfg)

print("OK")

TypeError: 'dict' object is not callable

In [1]:
from data import prepare

cfg["data"]["root"] = "/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL"

cache, manifest = prepare(cfg)

ModuleNotFoundError: No module named 'data'

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [3]:
import config
print(config.__file__)

/media/ehsan/MyFiles2/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src/config.py


In [4]:
from data import prepare
from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

cfg["data"]["root"] = "/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL"

cache, manifest = prepare(cfg)


KeyError: 'start'

In [6]:
import importlib
import config

importlib.reload(config)

from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")
print(cfg["data"])

{'root': 'PTB-XL', 'sampling_rate': 100, 'history': 700, 'horizon': 300, 'synthetic': False, 'start': 0, 'layout': 'time_first'}


In [7]:
cfg["data"]["root"] = "/media/ehsan/MyFiles2/FraUAS/HIS Project/PTB-XL"

from data import prepare

cache, manifest = prepare(cfg)

print(cache)

data/study_v3/ee7d4f585e459326


In [8]:
print(manifest["splits"])

{'train': {'records': 339, 'patients': 300, 'windows': 339}, 'val': {'records': 81, 'patients': 75, 'windows': 81}}


In [9]:
from data import ECGDataset
from preprocessing import feature_arrays

arrays, names, details = feature_arrays(
    cfg,
    cache,
    "F0",
    splits=("train", "val")
)

ds = ECGDataset(cache, cfg, "train", "P0", arrays["train"])

sample = ds[0]

print("x", sample["x"].shape)
print("y", sample["y"].shape)
print("static", sample["static"].shape)


x torch.Size([700, 12])
y torch.Size([300, 12])
static torch.Size([0])


In [10]:
from models import build_model

model = build_model(
    "patchtst",
    static_dim=0,
    horizon=cfg["data"]["horizon"],
    **cfg["model"]
)

print(model)

PatchTSTForecaster(
  (proj): Linear(in_features=12, out_features=12, bias=True)
)


In [11]:
import torch

x = torch.randn(2, 700, 12)
y = model(x)

print(y.shape)

torch.Size([2, 300, 12])


In [13]:
import einops
print(einops.__version__)

0.8.2


In [14]:
print(cfg["model"])

{'hidden': 64, 'depth': 1, 'dropout': 0.2, 'head_width': 128, 'static_embedding': 16}


In [15]:
from models import build_model

model = build_model(
    "patchtst",
    static_dim=0,
    horizon=300,
    **cfg["model"]
)

x = torch.randn(2, 700, 12)

y = model(x)

print(y.shape)

torch.Size([2, 300, 12])
